In [ ]:
import json
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from nba_api.stats.endpoints import scoreboardv2 as scoreboard
from backend.lib import NbaApiHelper as NbaHelper

GAMEHEADER = 0
LINESCORE = 1
SERIESSTANDINGS = 2
LASTMEETING = 3
EASTCONFSTANDINGSBYDAY = 4
WESTCONFSTANDINGSBYDAY = 5
AVAILABLE = 6
TEAMLEADERS = 7
TICKETLINKS = 8

In [6]:
def master_game_table(games, year):
    with_points = get_points(games)
    with_abbreviations_and_logos = get_abbreviations_and_logos( with_points, year)
    return with_abbreviations_and_logos

def get_points(games):
    merged = games[GAMEHEADER].merge(
        games[LINESCORE][["TEAM_ID", "PTS"]],
        left_on="HOME_TEAM_ID",
        right_on="TEAM_ID",
        how="left"
    ).rename(columns={"PTS": "HOME_SCORE"})

    # Merge away scores
    merged = merged.merge(
        games[LINESCORE][["TEAM_ID", "PTS"]],
        left_on="VISITOR_TEAM_ID",
        right_on="TEAM_ID",
        how="left"
    ).rename(columns={"PTS": "AWAY_SCORE"})

    merged.drop(columns=["TEAM_ID_x", "TEAM_ID_y"], inplace=True)
    return merged

def get_abbreviations_and_logos(games, year):
    games["HOME_ABBREVIATION"] = games.apply(
        lambda row: NbaHelper.get_team_abbreviation(row["HOME_TEAM_ID"]), axis=1
    )
    games["VISITOR_ABBREVIATION"] = games.apply(
        lambda row: NbaHelper.get_team_abbreviation(row["VISITOR_TEAM_ID"]), axis=1
    )
    games["HOME_LOGO"] = games.apply(
        lambda row: NbaHelper.create_logo_lookup(row["HOME_ABBREVIATION"], year),
        axis=1,
    )
    games["VISITOR_LOGO"] = games.apply(
        lambda row: NbaHelper.create_logo_lookup(row["VISITOR_ABBREVIATION"], year),
        axis=1,
    )
    return games

In [7]:
def get_current_games(year, month, day):
    master_games = scoreboard.ScoreboardV2(
        game_date=f"{year}-{month}-{day}", league_id="00", day_offset=0
    ).get_data_frames()
    games = master_game_table(master_games, year)
    return games

In [8]:
test_df = get_current_games(2024, 4, 10)
test_df

,GAME_DATE_EST,GAME_SEQUENCE,GAME_ID,GAME_STATUS_ID,GAME_STATUS_TEXT,GAMECODE,HOME_TEAM_ID,VISITOR_TEAM_ID,SEASON,LIVE_PERIOD,...,LIVE_PERIOD_TIME_BCAST,ARENA_NAME,WH_STATUS,WNBA_COMMISSIONER_FLAG,HOME_SCORE,AWAY_SCORE,HOME_ABBREVIATION,VISITOR_ABBREVIATION,HOME_LOGO,VISITOR_LOGO
0,2024-04-10T00:00:00,1,0022301158,3,Final,20240410/MEMCLE,1610612739,1610612763,2023,4,...,Q4 -,Rocket Mortgage FieldHouse,1,0,110,98,CLE,MEM,CLE-2024,MEM-2024
1,2024-04-10T00:00:00,2,0022301159,3,Final,20240410/CHAATL,1610612737,1610612766,2023,4,...,Q4 -,State Farm Arena,1,0,114,115,ATL,CHA,ATL-2024,CHA-2024
2,2024-04-10T00:00:00,3,0022301160,3,Final,20240410/TORBKN,1610612751,1610612761,2023,4,...,Q4 -,Barclays Center,1,0,106,102,BKN,TOR,NJN-2024,TOR-2024
3,2024-04-10T00:00:00,4,0022301161,3,Final,20240410/DALMIA,1610612748,1610612742,2023,4,...,Q4 - ESPN/ESPN2,Kaseya Center,1,0,92,111,MIA,DAL,MIA-2024,DAL-2024
4,2024-04-10T00:00:00,5,0022301162,3,Final,20240410/ORLMIL,1610612749,1610612753,2023,4,...,Q4 -,Fiserv Forum,1,0,117,99,MIL,ORL,MIL-2024,ORL-2024
5,2024-04-10T00:00:00,6,0022301163,3,Final,20240410/SASOKC,1610612760,1610612759,2023,4,...,Q4 -,Paycom Center,1,0,127,89,OKC,SAS,OKC-2024,SAS-2024
6,2024-04-10T00:00:00,7,0022301164,3,Final,20240410/MINDEN,1610612743,1610612750,2023,4,...,Q4 - ESPN,Ball Arena,1,0,116,107,DEN,MIN,DEN-2024,MIN-2024
7,2024-04-10T00:00:00,8,0022301165,3,Final,20240410/PHXLAC,1610612746,1610612756,2023,4,...,Q4 -,Crypto.com Arena,1,0,108,124,LAC,PHX,LAC-2024,PHO-2024


In [9]:
# Merge home scores
merged = games[GAMEHEADER].merge(
    games[LINESCORE][["TEAM_ID", "PTS"]],
    left_on="HOME_TEAM_ID",
    right_on="TEAM_ID",
    how="left"
).rename(columns={"PTS": "HOME_SCORE"})

# Merge away scores
merged = merged.merge(
    games[LINESCORE][["TEAM_ID", "PTS"]],
    left_on="VISITOR_TEAM_ID",
    right_on="TEAM_ID",
    how="left"
).rename(columns={"PTS": "AWAY_SCORE"})

merged.drop(columns=["TEAM_ID_x", "TEAM_ID_y"], inplace=True)
merged

NameError: name 'games' is not defined